## Setup & Imports

In [2]:
CITY_CKPT  = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_cityscapes.bin"
COCO_CKPT  = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco.bin"
DATA_PATH_CITY = "/content/drive/MyDrive/project_data_city"
DATA_PATH_COCO = "/content/drive/MyDrive/project_data_coco"
CITY_CONFIG_PATH = "/content/MaskArchitectureAnomaly_Project2026/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
COCO_CONFIG_PATH = "/content/MaskArchitectureAnomaly_Project2026/eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
FINET_CONFIG_PATH = '/content/MaskArchitectureAnomaly_Project2026/eomt/configs/dinov2/coco/panoptic/eomt_finetuning_step3.yaml'
FINET_CKPT = '/content/drive/MyDrive/checkpoints_task5/step3_best.ckpt'
# ↑ point to the folder containing the zip, not the zip itself

In [3]:
import yaml
import importlib
import warnings

import torch
import torch.nn.functional as F
from torch.amp.autocast_mode import autocast
from torchmetrics import JaccardIndex

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from lightning import seed_everything

seed_everything(0, verbose=False)
warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module`.*",
)

DEVICE     = 0          # indice GPU
IMG_IDX    = 0          # immagine da visualizzare

CITY_DATA_PATH = DATA_PATH_CITY
COCO_DATA_PATH = DATA_PATH_COCO
FINET_DATA_PATH = DATA_PATH_CITY
CITY_CKPT      = CITY_CKPT
COCO_CKPT      = COCO_CKPT
FINET_CKPT     = FINET_CKPT

# Config YAML forniti dal laboratorio
CITY_CONFIG_PATH = "/content/MaskArchitectureAnomaly_Project2026/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
COCO_CONFIG_PATH = "/content/MaskArchitectureAnomaly_Project2026/eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
FINET_CONFIG_PATH = "/content/MaskArchitectureAnomaly_Project2026/eomt/configs/dinov2/coco/panoptic/eomt_finetuning_step3.yaml"
# ──────────────────────────────────────────────────────────────────────────────

IGNORE_INDEX           = 255
NUM_CITYSCAPES_CLASSES = 19

## Load Model and Data

In [20]:
def load_model_and_data(config_path, ckpt_path, data_path, device):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    # Dataset
    data_module_name, class_name = config["data"]["class_path"].rsplit(".", 1)
    data_module = getattr(importlib.import_module(data_module_name), class_name)
    data_kwargs = config["data"].get("init_args", {})
    data_kwargs.pop("num_workers", None)
    data_kwargs.pop("path", None)
    data_kwargs.pop("batch_size", None)
    data = data_module(
        path=data_path,
        batch_size=1,
        num_workers=0,
        check_empty_targets=False,
        **data_kwargs,
    ).setup()


    # Encoder
    enc_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    enc_mod, enc_cls = enc_cfg["class_path"].rsplit(".", 1)
    encoder = getattr(importlib.import_module(enc_mod), enc_cls)(
        img_size=data.img_size, **enc_cfg.get("init_args", {})
    )

    # Network
    net_cfg = config["model"]["init_args"]["network"]
    net_mod, net_cls = net_cfg["class_path"].rsplit(".", 1)
    net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k not in ["encoder", "num_classes"]}
    network = getattr(importlib.import_module(net_mod), net_cls)(
        masked_attn_enabled=False,
        num_classes=data.num_classes,
        encoder=encoder,
        **net_kwargs,
    )

    # Lightning module
    lit_mod, lit_cls = config["model"]["class_path"].rsplit(".", 1)
    lit_cls = getattr(importlib.import_module(lit_mod), lit_cls)
    model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}
    if "stuff_classes" in config["data"].get("init_args", {}):
        model_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]
    model_kwargs.pop("num_classes", None)

    model = lit_cls(
        img_size=data.img_size,
        num_classes=data.num_classes,
        network=network,
        **model_kwargs,
    ).eval().to(device)

    # Weights
    ckpt = torch.load(ckpt_path, map_location=f"cuda:{device}", weights_only=False)
    model.load_state_dict(ckpt, strict=False)
    print(f"Loaded: {ckpt_path}")

    return model, data

## Unzip Datasets

In [6]:
!unzip "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets.zip" -d /content/datasets/

Archive:  /content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets.zip
   creating: /content/datasets/Validation_Dataset/
   creating: /content/datasets/Validation_Dataset/RoadAnomaly21/
  inflating: /content/datasets/Validation_Dataset/.DS_Store  
  inflating: /content/datasets/__MACOSX/Validation_Dataset/._.DS_Store  
   creating: /content/datasets/Validation_Dataset/RoadAnomaly/
   creating: /content/datasets/Validation_Dataset/RoadObsticle21/
   creating: /content/datasets/Validation_Dataset/fs_static/
   creating: /content/datasets/Validation_Dataset/FS_LostFound_full/
  inflating: /content/datasets/Validation_Dataset/RoadAnomaly21/.DS_Store  
  inflating: /content/datasets/__MACOSX/Validation_Dataset/RoadAnomaly21/._.DS_Store  
   creating: /content/datasets/Validation_Dataset/RoadAnomaly21/images/
   creating: /content/datasets/Validation_Dataset/RoadAnomaly21/labels_masks/
  inflating: /content/datasets/Validation_Dataset/RoadAnomaly/.DS_Store  
  inflating: /con

## EvalAnomalyEOMT.py
(inference + GPU accelerated metrics)

In [12]:
import os
import glob
import torch
import numpy as np
from PIL import Image
import torch.nn.functional as F
from torchvision.transforms import Compose, ToTensor
from tqdm import tqdm

from sklearn.metrics import average_precision_score
from ood_metrics import fpr_at_95_tpr

def evaluate_anomaly_eomt(model, data, images_dir, gt_dir, device='cuda:0'):
    # No Resize!
    input_transform = Compose([ToTensor()])
    model.eval()

    # Define the temperatures required for the table, plus a few extra for finding the best t
    temperatures = [0.1, 0.5, 0.75, 1.0, 1.1, 1.5, 2.0]

    ood_gts_list = []
    anomaly_scores = {
        "Max_Entropy": [],
        "MaxLogit": [],
        "RbA": []
    }

    # Dynamically add an MSP key for every temperature
    for t in temperatures:
        anomaly_scores[f"MSP (t={t})"] = []

    image_paths = glob.glob(os.path.join(images_dir, '*.png'))
    image_paths.extend(glob.glob(os.path.join(images_dir, '*.jpg')))
    image_paths.extend(glob.glob(os.path.join(images_dir, '*.webp')))

    print(f"Inizio inferenza su {len(image_paths)} immagini (Risoluzione Naturale)...")

    for img_path in tqdm(image_paths):
        # =============================================================
        # 1. GROUND TRUTH
        # =============================================================
        img_basename = os.path.basename(img_path)
        # Remove the original extention (.jpg, .png o .webp) forcing .png for the GT
        base_name_without_ext = os.path.splitext(img_basename)[0]
        gt_path = os.path.join(gt_dir, base_name_without_ext + '.png')

        if not os.path.exists(gt_path):
            continue

        gt_img = Image.open(gt_path).convert('L')
        gt_array = np.array(gt_img)

        # Original Mapping
        if "RoadAnomaly" in gt_path:
            gt_array = np.where((gt_array==2), 1, gt_array)
        elif "LostAndFound" in gt_path:
            gt_array = np.where((gt_array==0), 255, gt_array)
            gt_array = np.where((gt_array==1), 0, gt_array)
            gt_array = np.where((gt_array>1) & (gt_array<201), 1, gt_array)
        elif "Streethazard" in gt_path:
            gt_array = np.where((gt_array==14), 255, gt_array)
            gt_array = np.where((gt_array<20), 0, gt_array)
            gt_array = np.where((gt_array==255), 1, gt_array)

        if 1 not in np.unique(gt_array):
            continue

        ood_gts_list.append(gt_array.flatten())

        # =============================================================
        # 2. INFERENCE AND WINDOWING
        # =============================================================
        image = Image.open(img_path).convert('RGB')
        img_tensor = input_transform(image)
        img_tensor = (img_tensor * 255).to(torch.uint8)

        with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            imgs = [img_tensor.to(device)]
            img_sizes = [imgs[0].shape[-2:]]

            crops, origins = model.window_imgs_semantic(imgs)
            mask_logits_per_layer, class_logits_per_layer = model(crops)

            mask_logits = F.interpolate(mask_logits_per_layer[-1], size=crops.shape[-2:], mode="bilinear", align_corners=False)
            class_logits = class_logits_per_layer[-1]

            # =============================================================
            # 3. STANDARD METRICS WITH TEMPERATURE SCALING
            # =============================================================
            crop_probs = model.to_per_pixel_logits_semantic(mask_logits, class_logits)
            probs_list = model.revert_window_logits_semantic(crop_probs, origins, img_sizes)

            # Raw Logits
            probs_f32 = probs_list[0].float().cpu().numpy()

            # --- MaxLogit ---
            maxlogit_score = 1.0 - np.max(probs_f32, axis=0)
            anomaly_scores["MaxLogit"].append(maxlogit_score.flatten())

            # --- Temperature Scaling Loop ---
            for t in temperatures:
                # 1. Scale logits
                scaled_logits = probs_f32 / t

                # 2. Apply Softmax
                e_x = np.exp(scaled_logits - np.max(scaled_logits, axis=0, keepdims=True))
                softmax_probs = e_x / np.sum(e_x, axis=0, keepdims=True)

                # 3. Calculate MSP
                msp_score = 1.0 - np.max(softmax_probs, axis=0)
                anomaly_scores[f"MSP (t={t})"].append(msp_score.flatten())

                # 4. Calculate MaxEntropy ONLY on standard t=1.0
                if t == 1.0:
                    entropy_score = -np.sum(softmax_probs * np.log(softmax_probs + 1e-12), axis=0)
                    anomaly_scores["Max_Entropy"].append(entropy_score.flatten())

            # =============================================================
            # 4. RbA (Log-Sum Anti-Underflow)
            # =============================================================
            B, Q = mask_logits.shape[0], mask_logits.shape[1]

            mask_probs_f32 = mask_logits.sigmoid().float()
            class_probs_f32 = class_logits.softmax(dim=-1).float()

            prob_known_q = 1.0 - class_probs_f32[..., -1]
            joint_probs = mask_probs_f32 * prob_known_q.view(B, Q, 1, 1)

            rejection_probs = torch.clamp(1.0 - joint_probs, min=1e-7, max=1.0)
            log_rba_crop = torch.sum(torch.log(rejection_probs), dim=1, keepdim=True).half()

            rba_list = model.revert_window_logits_semantic(log_rba_crop, origins, img_sizes)
            rba_score = rba_list[0].squeeze(0).cpu().numpy()

            anomaly_scores["RbA"].append(rba_score.flatten())

            del image, img_tensor, imgs, crops, mask_logits, class_logits, crop_probs, log_rba_crop
            torch.cuda.empty_cache()

    # =============================================================
    # FINAL ANOMALY DETECTION AND OUTPUT TABLE (GPU ACCELERATED)
    # =============================================================
    print("\nElaborazione dati e calcolo AUC su GPU in corso...")
    ood_gts = np.concatenate(ood_gts_list)
    ood_mask = (ood_gts == 1)
    ind_mask = (ood_gts == 0)

    # GPU-accelerated metrics function (1000x faster than scikit-learn on CPU)
    def calculate_metrics_gpu(scores_array, device='cuda:0'):
        # 1. Move vectors to GPU
        scores_t = torch.tensor(np.concatenate((scores_array[ind_mask], scores_array[ood_mask])), dtype=torch.float32, device=device)
        labels_t = torch.cat([
            torch.zeros(ind_mask.sum(), dtype=torch.float32, device=device),
            torch.ones(ood_mask.sum(), dtype=torch.float32, device=device)
        ])

        # 2. Parallel Sort on GPU
        desc_scores, indices = torch.sort(scores_t, descending=True)
        desc_labels = labels_t[indices]

        # 3. Vectorized True Positives (TP) and False Positives (FP)
        tp = torch.cumsum(desc_labels, dim=0)
        fp = torch.cumsum(1.0 - desc_labels, dim=0)
        total_positives = desc_labels.sum()
        total_negatives = labels_t.size(0) - total_positives

        # --- AUPRC ---
        precision = tp / (tp + fp + 1e-12)
        recall = tp / total_positives

        recall_diff = torch.cat([recall[0:1], recall[1:] - recall[:-1]])
        prc_auc = torch.sum(recall_diff * precision).item() * 100.0

        # --- FPR@95TPR ---
        idx_95 = torch.where(recall >= 0.95)[0][0]
        fpr_95 = (fp[idx_95] / total_negatives).item() * 100.0

        # VRAM Cleanup
        del scores_t, labels_t, desc_scores, indices, desc_labels, tp, fp, precision, recall, recall_diff
        torch.cuda.empty_cache()

        return prc_auc, fpr_95

    # Collect results using GPU tracking
    final_results = {}
    for k in anomaly_scores:
        if len(anomaly_scores[k]) > 0:
            flat_scores = np.concatenate(anomaly_scores[k])
            final_results[k] = calculate_metrics_gpu(flat_scores, device=device)

    # Find the best MSP temperature (highest AUPRC)
    best_msp_t_name = ""
    best_msp_auprc = -1
    best_msp_fpr = -1

    for k, (auprc, fpr) in final_results.items():
        if "MSP" in k:
            if auprc > best_msp_auprc:
                best_msp_auprc = auprc
                best_msp_fpr = fpr
                best_msp_t_name = k.replace("MSP ", "")

    # --- FORMATTED CONSOLE OUTPUT ---
    print("\n" + "="*50)
    print(f"{'RISULTATI ANOMALY DETECTION':^50}")
    print("="*50)
    print(f"| {'Method':<18} | {'AUPRC (%)':<10} | {'FPR95 (%)':<10} |")
    print(f"|{'-'*20}|{'-'*12}|{'-'*12}|")

    # Define display order
    display_order = ["MSP (t=1.0)", "MSP (t=0.5)", "MSP (t=0.75)", "MSP (t=1.1)"]

    for method in display_order:
        if method in final_results:
            auprc, fpr = final_results[method]
            display_name = "MSP" if method == "MSP (t=1.0)" else method
            print(f"| {display_name:<18} | {auprc:>9.2f} | {fpr:>9.2f} |")

    # Print Best MSP dynamically
    print(f"| {'MSP (best t)':<18} | {best_msp_auprc:>9.2f} | {best_msp_fpr:>9.2f} |")
    print(f"|{'-'*20}|{'-'*12}|{'-'*12}|")

    # Print the rest of the standard metrics
    for method in ["MaxLogit", "Max_Entropy", "RbA"]:
        if method in final_results:
            auprc, fpr = final_results[method]
            print(f"| {method:<18} | {auprc:>9.2f} | {fpr:>9.2f} |")
    print("="*50)
    print(f"Nota: Il 'best t' trovato è stato {best_msp_t_name}\n")

## Cityscapes Model

In [ ]:
# ── Cityscapes ────────────────────────────────────────────────────────────────
city_model, city_data = load_model_and_data(
    config_path = CITY_CONFIG_PATH,
    ckpt_path   = CITY_CKPT,
    data_path   = DATA_PATH_CITY,
    device      = DEVICE,
)


print(f"\ncity  →  img_size={city_data.img_size}, num_classes={city_data.num_classes}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loaded: /content/drive/MyDrive/CourseProjectAnomaly/eomt_cityscapes.bin

city  →  img_size=(1024, 1024), num_classes=19


### RoadAnomaly

In [ ]:
evaluate_anomaly_eomt(
    model=city_model,
    data=city_data,
    images_dir='/content/datasets/Validation_Dataset/RoadAnomaly/images',
    gt_dir='/content/datasets/Validation_Dataset/RoadAnomaly/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 60 immagini (Risoluzione Naturale)...


100%|██████████| 60/60 [01:36<00:00,  1.60s/it]



Elaborazione dati e calcolo AUC in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |     71.38 |     15.48 |
| MSP (t=0.5)        |     71.16 |     15.69 |
| MSP (t=0.75)       |     71.29 |     15.53 |
| MSP (t=1.1)        |     71.40 |     15.47 |
| MSP (best t)       |     71.58 |     15.43 |
|--------------------|------------|------------|
| MaxLogit           |     70.81 |     15.09 |
| Max_Entropy        |     74.16 |     14.72 |
| RbA                |     70.27 |     17.94 |
Nota: Il 'best t' trovato è stato (t=2.0)



### SMIYC RoadAnomaly-21

In [ ]:
evaluate_anomaly_eomt(
    model=city_model,
    data=city_data,
    images_dir='/content/datasets/Validation_Dataset/RoadAnomaly21/images',
    gt_dir='/content/datasets/Validation_Dataset/RoadAnomaly21/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 10 immagini (Risoluzione Naturale)...


100%|██████████| 10/10 [00:11<00:00,  1.17s/it]



Elaborazione dati e calcolo AUC in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |     68.13 |     30.23 |
| MSP (t=0.5)        |     67.98 |     30.22 |
| MSP (t=0.75)       |     68.07 |     30.23 |
| MSP (t=1.1)        |     68.16 |     30.23 |
| MSP (best t)       |     68.30 |     30.23 |
|--------------------|------------|------------|
| MaxLogit           |     67.63 |     31.41 |
| Max_Entropy        |     68.31 |     30.42 |
| RbA                |     66.25 |     38.04 |
Nota: Il 'best t' trovato è stato (t=2.0)



### SMIYC RoadObstacle-21

In [ ]:
evaluate_anomaly_eomt(
    model=city_model,
    data=city_data,
    images_dir='/content/datasets/Validation_Dataset/RoadObsticle21/images',
    gt_dir='/content/datasets/Validation_Dataset/RoadObsticle21/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 30 immagini (Risoluzione Naturale)...


100%|██████████| 30/30 [01:18<00:00,  2.61s/it]



Elaborazione dati e calcolo AUC in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |     94.18 |      0.37 |
| MSP (t=0.5)        |     94.16 |      0.37 |
| MSP (t=0.75)       |     94.17 |      0.37 |
| MSP (t=1.1)        |     94.18 |      0.37 |
| MSP (best t)       |     94.18 |      0.37 |
|--------------------|------------|------------|
| MaxLogit           |     94.21 |      0.36 |
| Max_Entropy        |     94.27 |      0.35 |
| RbA                |     94.14 |      0.38 |
Nota: Il 'best t' trovato è stato (t=2.0)



### FS Lost&Found

In [ ]:
evaluate_anomaly_eomt(
    model=city_model,
    data=city_data,
    images_dir='/content/datasets/Validation_Dataset/FS_LostFound_full/images',
    gt_dir='/content/datasets/Validation_Dataset/FS_LostFound_full/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 100 immagini (Risoluzione Naturale)...


100%|██████████| 100/100 [04:36<00:00,  2.77s/it]



Elaborazione dati e calcolo AUC in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |     16.37 |     12.97 |
| MSP (t=0.5)        |     16.36 |     12.97 |
| MSP (t=0.75)       |     16.36 |     12.97 |
| MSP (t=1.1)        |     16.37 |     12.96 |
| MSP (best t)       |     16.39 |     12.96 |
|--------------------|------------|------------|
| MaxLogit           |     16.35 |     12.74 |
| Max_Entropy        |     19.07 |     12.79 |
| RbA                |     16.35 |     12.44 |
Nota: Il 'best t' trovato è stato (t=2.0)



### FS Static

In [ ]:
evaluate_anomaly_eomt(
    model=city_model,
    data=city_data,
    images_dir='/content/datasets/Validation_Dataset/fs_static/images',
    gt_dir='/content/datasets/Validation_Dataset/fs_static/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 30 immagini (Risoluzione Naturale)...


100%|██████████| 30/30 [00:40<00:00,  1.36s/it]



Elaborazione dati e calcolo AUC in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |     58.18 |     41.16 |
| MSP (t=0.5)        |     58.16 |     41.22 |
| MSP (t=0.75)       |     58.17 |     41.23 |
| MSP (t=1.1)        |     58.18 |     41.15 |
| MSP (best t)       |     58.19 |     41.13 |
|--------------------|------------|------------|
| MaxLogit           |     58.27 |     46.37 |
| Max_Entropy        |     56.82 |     41.85 |
| RbA                |     59.55 |     43.26 |
Nota: Il 'best t' trovato è stato (t=2.0)



## COCO Model

In [5]:
# ── COCO ──────────────────────────────────────────────────────────────────────

coco_model, coco_data = load_model_and_data(
    config_path = COCO_CONFIG_PATH,
    ckpt_path   = COCO_CKPT,
    data_path   = DATA_PATH_COCO,
    device      = DEVICE,
)

print(f"coco  →  img_size={coco_data.img_size}, num_classes={coco_data.num_classes}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loaded: /content/drive/MyDrive/CourseProjectAnomaly/eomt_coco.bin
coco  →  img_size=(640, 640), num_classes=133


### RoadAnomaly

In [8]:
evaluate_anomaly_eomt(
    model=coco_model,
    data=coco_data,
    images_dir='/content/datasets/Validation_Dataset/RoadAnomaly/images',
    gt_dir='/content/datasets/Validation_Dataset/RoadAnomaly/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 60 immagini (Risoluzione Naturale)...


100%|██████████| 60/60 [07:45<00:00,  7.76s/it]



Elaborazione dati e calcolo AUC in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |     18.86 |     91.84 |
| MSP (t=0.5)        |     18.80 |     91.99 |
| MSP (t=0.75)       |     18.84 |     91.87 |
| MSP (t=1.1)        |     18.87 |     91.83 |
| MSP (best t)       |     18.89 |     91.79 |
|--------------------|------------|------------|
| MaxLogit           |     19.03 |     91.58 |
| Max_Entropy        |     21.54 |     87.88 |
| RbA                |     22.16 |     94.68 |
Nota: Il 'best t' trovato è stato (t=2.0)



### SMIYC RoadAnomaly-21

In [9]:
evaluate_anomaly_eomt(
    model=coco_model,
    data=coco_data,
    images_dir='/content/datasets/Validation_Dataset/RoadAnomaly21/images',
    gt_dir='/content/datasets/Validation_Dataset/RoadAnomaly21/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 10 immagini (Risoluzione Naturale)...


100%|██████████| 10/10 [01:13<00:00,  7.37s/it]



Elaborazione dati e calcolo AUC in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |     38.88 |     77.50 |
| MSP (t=0.5)        |     38.60 |     78.20 |
| MSP (t=0.75)       |     38.81 |     77.63 |
| MSP (t=1.1)        |     38.91 |     77.48 |
| MSP (best t)       |     38.98 |     77.38 |
|--------------------|------------|------------|
| MaxLogit           |     39.37 |     77.00 |
| Max_Entropy        |     43.21 |     63.32 |
| RbA                |     30.16 |     89.31 |
Nota: Il 'best t' trovato è stato (t=2.0)



### SMIYC RoadObstacle-21

In [10]:
evaluate_anomaly_eomt(
    model=coco_model,
    data=coco_data,
    images_dir='/content/datasets/Validation_Dataset/RoadObsticle21/images',
    gt_dir='/content/datasets/Validation_Dataset/RoadObsticle21/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 30 immagini (Risoluzione Naturale)...


100%|██████████| 30/30 [08:47<00:00, 17.59s/it]



Elaborazione dati e calcolo AUC in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |      2.76 |     99.98 |
| MSP (t=0.5)        |      2.74 |     99.98 |
| MSP (t=0.75)       |      2.75 |     99.98 |
| MSP (t=1.1)        |      2.76 |     99.98 |
| MSP (best t)       |      2.76 |     99.98 |
|--------------------|------------|------------|
| MaxLogit           |      2.76 |     99.98 |
| Max_Entropy        |      8.34 |     99.98 |
| RbA                |      3.49 |    100.00 |
Nota: Il 'best t' trovato è stato (t=2.0)



### FS Lost&Found

In [11]:
evaluate_anomaly_eomt(
    model=coco_model,
    data=coco_data,
    images_dir='/content/datasets/Validation_Dataset/FS_LostFound_full/images',
    gt_dir='/content/datasets/Validation_Dataset/FS_LostFound_full/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 100 immagini (Risoluzione Naturale)...


100%|██████████| 100/100 [28:58<00:00, 17.38s/it]



Elaborazione dati e calcolo AUC in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |      3.78 |     97.21 |
| MSP (t=0.5)        |      3.77 |     97.22 |
| MSP (t=0.75)       |      3.78 |     97.21 |
| MSP (t=1.1)        |      3.78 |     97.21 |
| MSP (best t)       |      3.78 |     97.21 |
|--------------------|------------|------------|
| MaxLogit           |      3.78 |     97.21 |
| Max_Entropy        |      3.78 |     96.93 |
| RbA                |      2.41 |     60.68 |
Nota: Il 'best t' trovato è stato (t=2.0)



### FS Static

In [13]:
evaluate_anomaly_eomt(
    model=coco_model,
    data=coco_data,
    images_dir='/content/datasets/Validation_Dataset/fs_static/images',
    gt_dir='/content/datasets/Validation_Dataset/fs_static/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 30 immagini (Risoluzione Naturale)...


100%|██████████| 30/30 [04:24<00:00,  8.82s/it]



Elaborazione dati e calcolo AUC su GPU in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |      4.67 |     99.48 |
| MSP (t=0.5)        |      4.66 |     99.50 |
| MSP (t=0.75)       |      4.67 |     99.48 |
| MSP (t=1.1)        |      4.67 |     99.48 |
| MSP (best t)       |      4.67 |     99.47 |
|--------------------|------------|------------|
| MaxLogit           |      4.67 |     99.46 |
| Max_Entropy        |      4.57 |     99.32 |
| RbA                |      2.76 |     99.56 |
Nota: Il 'best t' trovato è stato (t=2.0)



## Finetuned Model

In [21]:
# ── Fine-Tuned COCO model (city_data) ────────────────────────────────────────────────────────
finetuned_model, finetuned_data = load_model_and_data(
    config_path = FINET_CONFIG_PATH,
    ckpt_path   = FINET_CKPT,
    data_path   = DATA_PATH_CITY,
    device      = DEVICE,
)

print(f"finetuned  →  img_size={finetuned_data.img_size}, num_classes={finetuned_data.num_classes}")

Loaded: /content/drive/MyDrive/checkpoints_task5/step3_best.ckpt
finetuned  →  img_size=[640, 640], num_classes=19


### RoadAnomaly

In [29]:
evaluate_anomaly_eomt(
    model=finetuned_model,
    data=finetuned_data,
    images_dir='/content/datasets/Validation_Dataset/RoadAnomaly/images',
    gt_dir='/content/datasets/Validation_Dataset/RoadAnomaly/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 60 immagini (Risoluzione Naturale)...


100%|██████████| 60/60 [00:53<00:00,  1.12it/s]



Elaborazione dati e calcolo AUC su GPU in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |     44.61 |     24.75 |
| MSP (t=0.5)        |     44.58 |     24.83 |
| MSP (t=0.75)       |     44.60 |     24.76 |
| MSP (t=1.1)        |     44.61 |     24.75 |
| MSP (best t)       |     44.62 |     24.75 |
|--------------------|------------|------------|
| MaxLogit           |     45.09 |     24.95 |
| Max_Entropy        |     46.47 |     25.20 |
| RbA                |     49.45 |     26.42 |
Nota: Il 'best t' trovato è stato (t=2.0)



### SMIYC RoadAnomaly-21

In [30]:
evaluate_anomaly_eomt(
    model=finetuned_model,
    data=finetuned_data,
    images_dir='/content/datasets/Validation_Dataset/RoadAnomaly21/images',
    gt_dir='/content/datasets/Validation_Dataset/RoadAnomaly21/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 10 immagini (Risoluzione Naturale)...


100%|██████████| 10/10 [00:08<00:00,  1.12it/s]



Elaborazione dati e calcolo AUC su GPU in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |     71.02 |     76.32 |
| MSP (t=0.5)        |     71.96 |     56.47 |
| MSP (t=0.75)       |     71.30 |     71.67 |
| MSP (t=1.1)        |     70.95 |     78.33 |
| MSP (best t)       |     74.93 |     25.66 |
|--------------------|------------|------------|
| MaxLogit           |     68.12 |     89.01 |
| Max_Entropy        |     67.72 |     98.47 |
| RbA                |     59.84 |     52.67 |
Nota: Il 'best t' trovato è stato (t=0.1)



### SMIYC RoadObstacle-21

In [31]:
evaluate_anomaly_eomt(
    model=finetuned_model,
    data=finetuned_data,
    images_dir='/content/datasets/Validation_Dataset/RoadObsticle21/images',
    gt_dir='/content/datasets/Validation_Dataset/RoadObsticle21/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 30 immagini (Risoluzione Naturale)...


100%|██████████| 30/30 [00:57<00:00,  1.92s/it]



Elaborazione dati e calcolo AUC su GPU in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |     94.96 |      0.19 |
| MSP (t=0.5)        |     94.89 |      0.20 |
| MSP (t=0.75)       |     94.94 |      0.19 |
| MSP (t=1.1)        |     94.97 |      0.19 |
| MSP (best t)       |     94.99 |      0.19 |
|--------------------|------------|------------|
| MaxLogit           |     95.13 |      0.16 |
| Max_Entropy        |     95.37 |      0.13 |
| RbA                |     94.59 |      0.15 |
Nota: Il 'best t' trovato è stato (t=2.0)



### FS Lost&Found

In [32]:
evaluate_anomaly_eomt(
    model=finetuned_model,
    data=finetuned_data,
    images_dir='/content/datasets/Validation_Dataset/FS_LostFound_full/images',
    gt_dir='/content/datasets/Validation_Dataset/FS_LostFound_full/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 100 immagini (Risoluzione Naturale)...


100%|██████████| 100/100 [03:57<00:00,  2.38s/it]



Elaborazione dati e calcolo AUC su GPU in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |     44.31 |     19.14 |
| MSP (t=0.5)        |     44.27 |     19.23 |
| MSP (t=0.75)       |     44.29 |     19.17 |
| MSP (t=1.1)        |     44.31 |     19.13 |
| MSP (best t)       |     44.32 |     19.09 |
|--------------------|------------|------------|
| MaxLogit           |     44.13 |     20.18 |
| Max_Entropy        |     43.86 |     18.82 |
| RbA                |     42.53 |     23.22 |
Nota: Il 'best t' trovato è stato (t=2.0)



### FS Static

In [33]:
evaluate_anomaly_eomt(
    model=finetuned_model,
    data=finetuned_data,
    images_dir='/content/datasets/Validation_Dataset/fs_static/images',
    gt_dir='/content/datasets/Validation_Dataset/fs_static/labels_masks',
    device='cuda:0'
)

Inizio inferenza su 30 immagini (Risoluzione Naturale)...


100%|██████████| 30/30 [00:37<00:00,  1.25s/it]



Elaborazione dati e calcolo AUC su GPU in corso...

           RISULTATI ANOMALY DETECTION            
| Method             | AUPRC (%)  | FPR95 (%)  |
|--------------------|------------|------------|
| MSP                |     84.79 |      6.69 |
| MSP (t=0.5)        |     84.72 |      6.65 |
| MSP (t=0.75)       |     84.77 |      6.69 |
| MSP (t=1.1)        |     84.79 |      6.69 |
| MSP (best t)       |     84.82 |      6.69 |
|--------------------|------------|------------|
| MaxLogit           |     85.23 |      6.60 |
| Max_Entropy        |     85.96 |      6.42 |
| RbA                |     87.44 |      6.05 |
Nota: Il 'best t' trovato è stato (t=2.0)

